
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Deploy a Pipeline with the Databricks CLI

In this lesson, you will learn how to manage pipelines using the Databricks CLI.

By the end of this lesson, you will be able to:
* Trigger a pipeline update
* Query a pipeline
* Clone a pipeline


## Run the setup
Run the setup script for this lesson by running the cell below. This will ensure that:
* The Databricks CLI is installed
* Authentication is configured
* A pipeline is created

In [0]:
%run ./Includes/Classroom-Setup-06.4

## Trigger a pipeline update

Use the following command to start the pipeline. Note that it uses an environment variable named **`DATABRICKS_PIPELINE_ID`** that was populated as part of the setup.

In [0]:
%sh databricks pipelines start-update $DATABRICKS_PIPELINE_ID --full-refresh


Use the following command to view the current status of the pipeline run. Repeat as needed.

In [0]:
%sh databricks pipelines get $DATABRICKS_PIPELINE_ID

## Clone a pipeline
Cloning a pipeline using the CLI involves getting the settings for a pipeline, removing elements of the settings that are not needed, changing the name of the pipeline, and creating a new pipeline with the changed settings. The commands below perform all of these actions.

Note the following:
* We use the **`get`** command to get the JSON output of the existing pipeline into a file named *settings.json*
* We process this output in a Python script that performs the following transformations:
    * Keeps only the **`spec`** portion of the configuration
    * Deletes the existing **`id`** (a new one will be created when the new pipeline is created)
    * Adjusts names for the pipeline itself and the target by appending with **`_copy`**

In [0]:
%sh
mkdir -p var && databricks pipelines get $DATABRICKS_PIPELINE_ID > var/settings.json
python << EOF 
import json

with open("var/settings.json", "r") as f:
    settings = json.load(f)['spec']

del settings['id']
settings['name'] = settings['name'] + '_copy'
settings['target'] = settings['target'] + '_copy'

with open("var/settings.json", "w") as f:
    json.dump(settings, f, indent=2)
EOF


### View settings

Use the following command to display the *settings.json* file. Since the file is created in your workspace, you could also display it through the workspace user interface, if desired.

In [0]:
%sh cat var/settings.json


### Create a new pipeline

Use the following command to create a new pipeline based on the *settings.json* file.

In [0]:
%sh databricks pipelines create --json @var/settings.json


### Run the new pipeline

Run the new pipeline, ensuring first that you copy the value for **`pipeline_id`** from the cell above into the cell below.

In [0]:
%sh databricks pipelines start-update "pipeline_id" --full-refresh


### Query status

Use the following command to view the current status of the cloned pipeline, again substituting **`pipeline_id`**. Repeat as needed.

In [0]:
%sh databricks pipelines get "pipeline_id"


### Delete the pipeline

Delete the cloned pipeline with the following command, again substituting **`pipeline_id`**.

In [0]:
%sh databricks pipelines delete "pipeline_id"


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>